# 6.390 Spring 2023 Homework 3

**If you haven't already, please hit :**

`File` -> `Save a Copy in Drive`

**to copy this notebook to your Google drive, and work on a copy. If you don't do this, your changes won't be saved!**

In [3]:
# Run this cell to download the test functions for HW 3
!wget --quiet --no-check-certificate https://introml.mit.edu/_static/spring26/homework/hw03/hw03_tests.py
from hw03_tests import *

import numpy as np

In [ ]:
import numpy as np
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler


def rv(value_list):
    """
    Takes a list of numbers and returns a row vector: 1 x n
    """
    return np.array([value_list])


def cv(value_list):
    """
    Takes a list of numbers and returns a column vector:  n x 1
    """
    return np.transpose(rv(value_list))


def lin_reg(X, th, th0):
    """
    Gets predictions from linear regression

    x (np.array): n-by-d
    th (np.array): d-by-m where m is the number of different settings of th
    th0 (np.array): m-by-1
    Returns m-by-n np.array of predictions
    """
    return np.dot(X, th) + th0


def mse(x, y, th, th0):
    return np.mean((y - lin_reg(x, th, th0)) ** 2, axis=0, keepdims=True)


def ridge_obj(x, y, th, th0, lam):
    return mse(x, y, th, th0) + lam * np.linalg.norm(th) ** 2


def ridge_analytic(X, y, lam):
    """
    Determine best theta using the analytic (closed-form) expression for ridge regression.

    X (np.array): n-by-d array of features
    y (np.array): n-by-1 array of targets
    lam (float): lambda i.e. L2-regularizer coefficient

    Returns: d-by-1 array of theta values
    """
    n, d = X.shape
    return np.linalg.inv(X.T @ X + n * lam * np.identity(d)) @ X.T @ y


def cross_validate_analytic(X, y, lam):
    n, d = X.shape
    total_loss = 0
    kf = KFold(n_splits=5)

    for train_index, test_index in kf.split(X, y=y):
        X_train_split, y_train_split = X[train_index], y[train_index]
        theta = ridge_analytic(X_train_split, y_train_split, lam)
        X_val_split, y_val_split = X[test_index], y[test_index]
        val_loss = np.mean((X_val_split @ theta - y_val_split) ** 2)
        total_loss += val_loss
    return total_loss / kf.n_splits


# Pre-Processing the data
# returning X_train, y_train, X_test, y_test.
# Train/Test split is 80/20.
def get_data_splits_with_transforms(data, selected_features, output_feature):
    # data is an n x d matrix
    # selected_features is a list of strings identifying the features we want to select
    # output_feature is the target variable (string)

    X = data[selected_features].values
    y = data[output_feature].values

    # Transform the data such that each feature lies in [-1, 1]
    scaler_x = StandardScaler().fit(X)
    X = scaler_x.transform(X)

    scaler_y = StandardScaler().fit(y)
    y = scaler_y.transform(y)

    # Create a training/test split.
    train_proportion = 0.8
    test_index = int(train_proportion * X.shape[0])

    X_test = np.copy(X[test_index:, :])
    X_train = np.copy(X[0:test_index, :])

    y_test = np.copy(y[test_index:, :])
    y_train = np.copy(y[0:test_index, :])

    return (X_train, y_train, X_test, y_test)


### Testing code


def TestCase(inputs, expected, fn=None):
    """
    Runs a single test case.
    """
    result = inputs[0]
    if fn:
        result = fn(*inputs)

    shape_error = False
    try:
        if np.allclose(result, expected, atol=1e-3):
            if isinstance(expected, np.ndarray):
                assert np.array_equal(expected.shape, result.shape)
            print("Test case passed!")
            print("----------------------------")
            return True
    except AssertionError:
        shape_error = True
    except:
        pass

    # display single inputs in a nicer way
    if len(inputs) == 1:
        inputs = inputs[0]

    if not fn:
        # don't show function inputs if not testing a function
        print("FAILED")
    else:
        print("FAILED on inputs:\n", inputs)

    print("Expected:\n", expected)
    print("Got:\n", result)

    if shape_error:
        print("Hint: Check the shapes of your output.")
    print("----------------------------")
    return False


def RunTestSuite(cases, fn=None, seed=None):
    """
    Cases is a list of lists (or other 2D iterable) where:
        * each row is a list/tuple/iterable, representing a test case
        * last element of each row is the expected answer
        * all elements before are the inputs to the function to check, in order
        * if the last element (expected answer) is a np array object, the
          student's answer will also be checked for getting shape correct.

    fn is the function to check. if fn=None, only the first element of Cases
       will be checked to see if it is np.allclose() to the last element.
    """
    passed = 0
    failed = 0
    for entries in cases:
        if seed is not None:
            np.random.seed(seed)
        expected = entries[-1]
        input = entries[:-1]
        if TestCase(input, expected, fn):
            passed += 1
        else:
            failed += 1
    if passed == len(cases):
        print("\nAll tests passed!")
    else:
        print("\nRan %d tests: %d passed, %d failed." % (len(cases), passed, failed))


def f1(x):
    return x[1:2, :] ** 2 + x[2:3, :]


def df1(x):
    x = list(x.squeeze())
    return cv([0, 2 * x[1], 1])


def f2(x):
    return x[0:1, :] * x[1:2, :]


def df2(x):
    x = list(x.squeeze())
    return cv([x[1], x[0], 1])


def package_ans(gd_vals):
    x, f = gd_vals
    return [x.tolist(), f.tolist()]


def TestGradientDescent(fn):
    expected1 = [
        [[1.0], [1.2302319221611202e-97], [-98.99999999999865]],
        [[-98.99999999999865]],
    ]
    actual1 = fn(f1, df1, cv([1.0, 1.0, 1.0]), lambda x: 0.1, 1000)
    print("Test 1:")
    if np.allclose(expected1[0], actual1[0].tolist()) and np.all(
            abs(actual1[1] - expected1[1]) < 1.0e-4
    ):
        print("Passed!")
    else:
        print("Test 1 failed with f(x) = x[1:2, :]**2 + x[2:3, :]")
    expected2 = [
        [[-10479.577710978716], [10479.577926834949], [-5.999999999999876]],
        [[-109821551.26252407]],
    ]
    actual2 = fn(f2, df2, cv([2.0, 3.0, 4.0]), lambda x: 0.01, 1000)
    print("Test 2:")
    if np.allclose(actual2[0].tolist(), expected2[0]) and np.all(
            abs(actual2[1] - expected2[1]) < 1.0e-4
    ):
        print("Passed!")
    else:
        print("Test 2 failed with f(x) = x[0:1, :]*x[1:2, :]")


def TestNumGrad(fn):
    cases = [
        (
            f1,
            cv([1.0, 1.0, 1.0]),
            np.array(([[0.0], [2.0000000000000018], [0.9999999999998899]])),
        ),
        (
            f1,
            cv([1.0, 2.0, 3.0]),
            np.array(([[0.0], [3.9999999999995595], [0.9999999999994458]])),
        ),
        (
            f2,
            cv([-1.0, -1.0, -1.0]),
            np.array(([[-0.9999999999999454], [-0.9999999999999454], [0.0]])),
        ),
        (
            f2,
            cv([-1.0, -2.0, -3.0]),
            np.array(([[-1.9999999999998908], [-0.9999999999998899], [0.0]])),
        ),
    ]
    RunTestSuite(cases, lambda f, x: fn(f)(x))


def TestMinimize(fn):
    print("Test 1")
    ans = package_ans(fn(f1, cv([1.0, 1.0, 1.0]), lambda i: 0.1, 1000))
    expected = [
        [[1.0], [2.3564483697668948e-14], [-99.00000000010797]],
        [[-99.00000000010797]],
    ]
    if np.allclose(ans[0], expected[0]) and np.allclose(ans[1], expected[1]):
        print("Passed")
    else:
        print("Test 1 failed with f(x) = x[1:2, :]**2 + x[2:3, :]")

    print("Test 2")
    ans = package_ans(fn(f2, cv([2.0, 3.0, 4.0]), lambda i: 0.01, 1000))
    expected = [
        [[-10479.577713756074], [10479.57792981472], [4.0]],
        [[-109821551.32285637]],
    ]
    if np.allclose(ans[0], expected[0]) and np.allclose(ans[1], expected[1]):
        print("Passed")
    else:
        print("Test 2 failed with f(x) = x[0:1, :]*x[1:2, :]")


def TestSGD(sgd_func, num_grad):
    """
    Test for stochastic gradient descent sgd()

    Requires two arguments: your `sgd` function and your `make_num_grad_fn` function
    """

    def J(Xi, yi, w):
        # translate from (1-augmented X, y, theta) to (separated X, y, th, th0) format
        return ridge_obj(Xi[:, :-1], yi, w[:-1, :], w[-1:, :].T, 0).T

    def dJ(Xi, yi, w):
        def f(w):
            return J(Xi, yi, w)

        return num_grad(f)(w)

    X = np.array(
        [
            [0.0, 0.1, 0.2, 0.3, 0.42, 0.52, 0.72, 0.78, 0.84, 1.0],
            [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0],
        ]
    ).T
    y = np.array([[0.4, 0.6, 1.2, 0.1, 0.22, -0.6, -1.5, -0.5, -0.5, 0.0]]).T

    cases = [
        (
            X,
            y,
            J,
            dJ,
            cv([0.0, 0.0]),
            lambda i: 0.1,
            1000,
            np.array([[-1.438266235911751], [0.7366695997841343]]),
        ),
        (
            X,
            y,
            J,
            dJ,
            cv([-0.1, 0.1]),
            lambda i: 0.01,
            1000,
            np.array([[-1.1713461515169397], [0.5626086484724836]]),
        ),
    ]
    RunTestSuite(cases, sgd_func, seed=0)


def TestObjectiveFunc(objective_func):
    X_1 = np.array([[5, 5, 10, 16, 0]]).T
    Y_1 = np.array([[16, 14, 5, 1, 10]]).T

    X_2 = np.array([[6, 18, 12, 19, 15], [4, 7, 3, 10, 12], [16, 8, 15, 4, 6]]).T
    Y_2 = np.array([[0, 6, 15, 10, 16]]).T

    theta_1 = np.array([[3]])
    theta_2 = np.array([[5], [7], [3]])
    J_1_noreg = objective_func(X_1, Y_1, 0)(theta_1)
    assert isinstance(J_1_noreg, float), "Wrong return type"

    cases = [
        (X_1, Y_1, 0, theta_1, 587.2),
        (X_1, Y_1, 1, theta_1, 596.2),
        (X_2, Y_2, 2, theta_2, 20569.2),
    ]
    RunTestSuite(cases, lambda x, y, lam, th: objective_func(x, y, lam)(th))


def TestObjectiveFuncGrad(objective_func_grad):
    X_1 = np.array([[5, 5, 10, 16, 0]]).T
    Y_1 = np.array([[16, 14, 5, 1, 10]]).T

    X_2 = np.array([[6, 18, 12, 19, 15], [4, 7, 3, 10, 12], [16, 8, 15, 4, 6]]).T
    Y_2 = np.array([[0, 6, 15, 10, 16]]).T

    theta_1 = np.array([[3]])
    theta_2 = np.array([[5], [7], [3]])

    cases = [
        (X_1, Y_1, 0, theta_1, np.array([[400.8]])),
        (X_1, Y_1, 1, theta_1, np.array([[406.8]])),
        (X_2, Y_2, 2, theta_2, np.array([[4172.8], [2211.2000000000003], [2512.4]])),
    ]
    RunTestSuite(cases, lambda x, y, lam, th: objective_func_grad(x, y, lam)(th))


def TestRidgeGD(fn):
    X_train = np.array(
        [
            [2.367446386606739, 8.7520044953452, 8.14485148030185, 6.516389972287224],
            [
                0.039733614908928905,
                3.68480648450503,
                5.42213838170197,
                6.672864650109663,
            ],
            [
                3.9301662224536074,
                0.6049574778006783,
                1.7312268364620098,
                0.8299188668622759,
            ],
        ]
    ).T
    y_train = np.array(
        [[0.31776829626931735, 7.180994658970237, 2.402211421436035, 1.300292687028518]]
    ).T
    theta_actual = [[1.2447193743504172], [-1.0131608964646763], [-0.7209261159877435]]
    cases = [(X_train, y_train, 0.01, lambda i: 0.01, theta_actual)]
    RunTestSuite(cases, fn)


#### 10.2 Finding the Best Parameters (Optional)

Let's load the Boston Housing dataset. Our goal is to build a linear regression model (with regularization) to predict the TARGET_VAL (which is the median value of owner-occupied homes) using all other available features in the dataset.

For more information about the Boston housing dataset, please visit this [link](https://www.cs.toronto.edu/~delve/data/boston/bostonDetail.html).

Note that the data pre-processing routine below normalizes each feature. You will learn more about Feature transformations in Week 5.

In what follows, we use Cross-Validation to select the best hyperparameters for gradient descent on the ridge regression model. Using the best hyperparameters, we will then make predictions on a reserved test set. You will also compare the results when using the gradient descent based implementation vs the analytic (closed form) solution.

In [ ]:
## DO NOT EDIT BELOW.
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.model_selection import KFold

# Pre-Processing the data and returning the train and test sets.

# load the dataset and do some data exploration
data_url = "http://lib.stat.cmu.edu/datasets/boston"
raw_df = pd.read_csv(data_url, sep="\s+", skiprows=22, header=None)
data = np.hstack([raw_df.values[::2, :], raw_df.values[1::2, :2]])
target = raw_df.values[1::2, 2]

raw_data = np.concatenate((data, target[:, None]), axis=1)
xvars = [
    "CRIM",
    "ZN",
    "INDUS",
    "CHAS",
    "NOX",
    "RM",
    "AGE",
    "DIS",
    "RAD",
    "TAX",
    "PTRATIO",
    "B",
    "LSTAT",
]
yvars = ["TARGET_VAL"]

data = pd.DataFrame(raw_data, columns=xvars + yvars)

# Get the train and test splits to be used later.
X_train, y_train, X_test, y_test = get_data_splits_with_transforms(data, xvars, yvars)

**CODE REQUIRED HERE** Before we start using the Boston Housing dataset, let's implement the `ridge_gd` function. Given an input `X_train`, `y_train`, `lam`, `theta`, `step_size_fn`, and `num_steps`, run gradient descent on `X_train` and `y_train` starting from `theta = (dx1) vector of zeros`. Return the value of $\theta$ after running `num_steps` iterations of gradient descent.

```
inputs:
  X_train: a dxn numpy array
  y_train: a 1xn numpy array
  lam: lambda
  step_size_fn: a function that takes in i, the current training iteration, and returns the step size for iteration i
  num_steps: number of iterations

outputs:
  theta: value of theta after num_steps iterations of gradient descent
```
**Hint**: Your implementations of `objective_func` and `objective_func_grad` are very useful here! \
**Hint**: You can also use your `gd` function \
**Hint**: Previously, you've minimized f as a function of x. Now, X and y are constant. What variable are you minimizing over now? \

In [ ]:
def ridge_gd(X_train, y_train, lam, step_size_fn, num_steps=2000):
    # TODO
    # hint: start from theta = (dx1) vector of zeros

In [ ]:
TestRidgeGD(ridge_gd)

**CODE REQUIRED HERE** In `cross_validate_gd`, run 5-fold cross-validation on the X and y dataset. Use gradient descent to train a linear model for the `X`, `y` data. We've provided a for loop that iterates over each split. In this code:

```
  X_train_split, y_train_split: data to use for training. This is a (d x n) numpy array, where n=the number of datapoints in k-1 folds
  X_val_split, y_val_split: data to use for evaluating the model. This is a (d x n) numpy array, where n=the number of datapoints in 1 fold
```

**Hint**: Use `ridge_gd` here. \
**Hint**: Take a look at the solutions for last week's cross_validate code if you get stuck

In [ ]:
def cross_validate_gd(X, y, lam, step_size_fn):
    """
    Returns k-fold cross-validation loss. On each of the k folds,
    train a linear regression model using gradient descent. Return
    the average loss across the k folds.
    """
    total_loss = 0
    kf = KFold(n_splits=5)
    for train_index, test_index in kf.split(X, y=y):
        X_train_split, y_train_split = X[train_index].T, y[train_index].T
        # TODO - train model on X_train_split, y_train_split using gradient descent
        # hint - use variables step_size_fn and lam
        # TODO - evaluate model on X_val_split, y_val_split, add loss to total_loss
    return total_loss / kf.n_splits

Now it's time to run grid search! We are interested in running grid search over $\lambda \in \{{1e-4, 1e-3, \cdots, 1e-1\}}$ and $\eta \in \{{1e-6, 1e-5, \cdots, 1e-2\}}$.

These two cells are ready to run if you've correctly implemented `cross_validate_gd`. Use the outputs of these cells to answer the rest of problem 5.2.

We've also already implemented `cross_validate_analytic` for you. This function returns the cross-validation loss for linear regression models trained with the analytic solution for the squared loss equation.

**Note: The next two cells print the cross-validation loss, not the testing set loss! Run the last cell in this notebook for the testing set loss.**


In [ ]:
learning_rates = [1e-2, 1e-3, 1e-4, 1e-5, 1e-6]
lams = [1e-1, 1e-2, 1e-3, 1e-4]

# This code runs grid search over the training parameters in `learning_rates` and `lams`
for rate in learning_rates:
    for lam in lams:
        learning_rate_fn = lambda i: rate  # learning rate = `rate` throughout training
        cross_validation_loss = cross_validate_gd(
            X_train, y_train, lam, learning_rate_fn
        )
        print(
            f"Loss on dataset with lambda={lam}, rate={rate} : cross_validation_loss {cross_validation_loss:.6f}"
        )

In [ ]:
lams = [1e-1, 1e-2, 1e-3, 1e-4]

# This code runs grid search over the training parameters in `lams`
for lam in lams:
    cross_validation_loss = cross_validate_analytic(X_train, y_train, lam).item()
    print(
        f"Loss on dataset with lambda={lam}: cross_validation_loss {cross_validation_loss:.6f}"
    )

We will now use the best params found above to build a model on the entire training set (X_train, y_train), get the $\theta$ values and use them to make predictions for the test set (X_test) and evaluate the error using the *actual* values (y_test). We will compare this error for the gradient descent based implementation vs the analytic solution.


**CODE REQUIRED HERE**:

1. Update **best_lam_gd** and **best_rate_gd** using the best $\lambda$ and $\eta$ values you found using **cross_validate_gd**() above.

2. Update **best_lam_analytic** using the best $\lambda$ value found by using **cross_validate_analytic**() above.



In [ ]:
## DO NOT EDIT THESE FUNCTIONS.
# Returns the test set predictions and error for the GD based implementation.
def get_gd_predictions_and_error(
    objective_func,
    objective_func_grad,
    gd_func,
    X_train,
    y_train,
    X_test,
    y_test,
    best_lam_gd,
    best_rate_gd,
):
    num_steps = 5000
    theta_gd = ridge_gd(
        X_train.T, y_train.T, best_lam_gd, lambda i: best_rate_gd, num_steps=num_steps
    )
    gd_predictions = theta_gd.T @ X_test.T
    gd_error = np.mean((gd_predictions - y_test) ** 2)
    return gd_predictions, gd_error


# Returns the test set predictions and error using the Analytic expression.
def get_analytic_predictions_and_error(X_train, y_train, X_test, y_test, best_lam):
    theta_analytic = ridge_analytic(X_train.T, y_train.T, best_lam)
    analytic_predictions = theta_analytic.T @ X_test.T
    analytic_error = np.mean((analytic_predictions - y_test) ** 2)
    return analytic_predictions, analytic_error

In [ ]:
#### Using the functions above along with the best performing hyperparams to
### determine the test set errors. Please specify the best lambda and learning
### rates for the GD and Analytic cases that you found above.

# GD
best_lam_gd = None  ### TODO: to be specified
best_rate_gd = None  ### TODO: to be specified

# get_gd_predictions() function is defined in the hw03 code you imported at
# the very top. Check the code out if you are curious about the implementation.
gd_predictions, gd_error = get_gd_predictions_and_error(
    objective_func,
    objective_func_grad,
    gd,
    X_train,
    y_train,
    X_test,
    y_test,
    best_lam_gd,
    best_rate_gd,
)

# Analytic
best_lam_analytic = None  ### TODO: to be specified

# get_analytic_predictions_and_error() function is defined in the hw03 code
# you imported at the very top. Check the code out if you are curious about the
# implementation.
analytic_predictions, analytic_error = get_analytic_predictions_and_error(
    X_train, y_train, X_test, y_test, best_lam_analytic
)


print(f"Test loss for GD based implementation={gd_error:0.3f}")
print(f"Test loss for Analytic (closed form) implementation={analytic_error:0.3f}")


#### (Optional) Compare the results by viewing the scatter plots for predictions.
plt.scatter(y_test, gd_predictions, color="red", label="GD")
plt.scatter(y_test, analytic_predictions, color="blue", label="Analytic")
plt.xlabel("Actual")
plt.ylabel("Predictions")
plt.title("Predictions vs Actual Scatter Plot")
plt.xlim([-3, 3])
plt.ylim([-3, 3])
plt.legend(loc="upper right")
plt.show()